# MeetMind — WhisperX Transcription Server

Chạy từng cell theo thứ tự. Sau **Cell 4** sẽ có URL để paste vào Railway.

> **Yêu cầu:** Đổi Runtime → T4 GPU trước khi chạy  
> Runtime → Change runtime type → T4 GPU

In [ ]:
# Cell 1 — Install dependencies
# Restart the KERNEL (not the VM) after install so the pinned numpy is reloaded.
# IMPORTANT: this only restarts the Python kernel — pip-installed packages stay on
# disk. Do NOT use runtime.unassign(): that recycles the whole VM and wipes the
# installs (whisperx disappears -> "No module named 'whisperx'" in Cell 2).
!pip install -q --force-reinstall "ctranslate2==4.5.0" "pandas==2.2.3" "setuptools<82" "numpy==2.2.2"
!pip install -q whisperx fastapi uvicorn pyngrok nest_asyncio httpx
print("Done installing — restarting KERNEL now (packages stay on disk)...")
print("After it reconnects, run Cell 2 onward. Do NOT re-run this cell.")

# Restart only the kernel; the VM and its installed packages persist.
import os
os.kill(os.getpid(), 9)


In [ ]:
# Cell 2 — Load WhisperX model (~2 phút lần đầu)
import torch, whisperx

device = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "int8"

print(f"Device: {device}")
model = whisperx.load_model("large-v2", device, compute_type=compute_type)
print("Model loaded!")

In [ ]:
# Cell 3 — Định nghĩa FastAPI /transcribe endpoint
import nest_asyncio, tempfile, os, httpx
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

nest_asyncio.apply()
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

class TranscribeRequest(BaseModel):
    audio_url: str

@app.get('/health')
def health():
    return {'status': 'ok'}

@app.post('/transcribe')
async def transcribe(req: TranscribeRequest):
    async with httpx.AsyncClient(timeout=120) as client:
        r = await client.get(req.audio_url)
    if r.status_code != 200:
        raise HTTPException(status_code=400, detail='Cannot download audio')

    suffix = '.m4a' if '.m4a' in req.audio_url else '.mp3'
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
        f.write(r.content)
        tmp = f.name

    try:
        audio = whisperx.load_audio(tmp)
        result = model.transcribe(audio, batch_size=16)
    finally:
        os.unlink(tmp)

    segments = [
        {'start': round(s['start'], 2), 'end': round(s['end'], 2), 'text': s['text'].strip()}
        for s in result['segments']
    ]
    return {'segments': segments, 'language': result.get('language', '')}

print("Server defined!")

In [ ]:
# Cell 4 — Expose qua ngrok + chạy server
# Lấy token miễn phí: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = ""  # <-- paste token vào đây

from pyngrok import ngrok
import uvicorn, threading, time, asyncio, httpx

if not NGROK_TOKEN:
    raise ValueError("Paste NGROK_TOKEN vào đây!")

ngrok.set_auth_token(NGROK_TOKEN)
ngrok.kill()  # kill tunnel cũ nếu có

# Run uvicorn in a background thread. install_signal_handlers MUST be disabled:
# uvicorn's default signal handlers can only be set from the main thread, so in a
# worker thread the server raises and dies silently -> "connection refused" on :8001.
config = uvicorn.Config(app, host="0.0.0.0", port=8001, log_level="info")
server = uvicorn.Server(config)
server.install_signal_handlers = lambda: None

def run():
    asyncio.run(server.serve())

threading.Thread(target=run, daemon=True).start()

# Wait until the server actually accepts connections before opening the tunnel.
for _ in range(40):
    try:
        if httpx.get("http://localhost:8001/health", timeout=1).status_code == 200:
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Server failed to start on :8001 — scroll up for the traceback")

tunnel = ngrok.connect(8001)
public_url = tunnel.public_url

print('=' * 60)
print(f"COLAB_WHISPER_URL = {public_url}/transcribe")
print('=' * 60)
print("Copy URL trên → paste vào backend/.env (COLAB_WHISPER_URL) → restart uvicorn")
print("Server running!")


In [ ]:
# Cell 5 (tuỳ chọn) — Test health check
import httpx
r = httpx.get(f"{public_url}/health")
print(r.json())  # {'status': 'ok'}